In [257]:
# !pip install transliterate
# !pip install openpyxl

In [258]:
import pandas as pd
import glob
import os
import numpy as np

In [259]:
df = pd.read_csv('zoki_liga_final.csv')

In [260]:
df.head(5)

,Број на играчи,Име,Позиција,Број на натпревари,Голови,Асистенции,Г/А,Г/А на натпревар,Хет-трик,Хет-трик од асистенции,...,Пенали,Спасени пенали,Победи,% на победи,Порази,% на порази,Играч на натпреварот,Номинации за играч на месец,Играч на месец,Гол на месецот
0,1,Дацо,D/GK,36,5,11,16,0.44,0,0,...,0/0,0/1,13,36%,16,44%,3,1,1,0
1,2,Лео,D/GK,34,40,19,59,1.74,3,3,...,0/1,1/2,12,35%,16,47%,3,2,3,1
2,3,Симиќ,AM,34,10,24,34,1.00,0,1,...,0/0,0/0,14,41%,16,47%,0,1,0,0
3,4,Стеф,M/GK,32,22,12,34,1.06,2,0,...,0/1,1/1,14,44%,12,38%,2,2,0,0
4,5,Нено,D/GK,29,0,6,6,0.21,0,0,...,0/1,1/3,11,38%,12,41%,0,2,0,0


In [261]:
CYRILLIC_TO_LATIN_MAP = {
    'А': 'A', 'а': 'a',
    'Б': 'B', 'б': 'b',
    'В': 'V', 'в': 'v',
    'Г': 'G', 'г': 'g',
    'Д': 'D', 'д': 'd',
    'Ѓ': 'Gj', 'ѓ': 'gj',
    'Е': 'E', 'е': 'e',
    'Ж': 'Zh', 'ж': 'zh',
    'З': 'Z', 'з': 'z',
    'И': 'I', 'и': 'i',
    'Ј': 'J', 'ј': 'j',
    'К': 'K', 'к': 'k',
    'Л': 'L', 'л': 'l',
    'Љ': 'Lj', 'љ': 'lj',
    'М': 'M', 'м': 'm',
    'Н': 'N', 'н': 'n',
    'Њ': 'Nj', 'њ': 'nj',
    'О': 'O', 'о': 'o',
    'П': 'P', 'п': 'p',
    'Р': 'R', 'р': 'r',
    'С': 'S', 'с': 's',
    'Т': 'T', 'т': 't',
    'Ќ': 'Kj', 'ќ': 'kj',
    'У': 'U', 'у': 'u',
    'Ф': 'F', 'ф': 'f',
    'Х': 'H', 'х': 'h',
    'Ц': 'C', 'ц': 'c',
    'Ч': 'Ch', 'ч': 'ch',
    'Џ': 'Dzh', 'џ': 'dzh',
    'Ш': 'Sh', 'ш': 'sh'
}
def cyrillicToLatinName(nameCyrillic):
    return ''.join(CYRILLIC_TO_LATIN_MAP.get(char, char) for char in nameCyrillic)

In [262]:
df = df.copy()

In [263]:
columns = df.columns.tolist()
columns

['Број на играчи',
 'Име',
 'Позиција',
 'Број на натпревари',
 'Голови',
 'Асистенции',
 'Г/А',
 'Г/А на натпревар',
 'Хет-трик ',
 'Хет-трик од асистенции',
 'Автоголови',
 'Слободни',
 'Пенали',
 'Спасени пенали ',
 'Победи',
 '% на победи',
 'Порази',
 '% на порази',
 'Играч на натпреварот',
 'Номинации за играч на месец',
 'Играч на месец',
 'Гол на месецот']

In [264]:
columns_translating_mk_to_en = {
'Број на играчи' : 'index',
 'Име':'name',
 'Позиција':'position',
 'Број на натпревари':'matches',
 'Голови':'goals',
 'Асистенции':'assists',
 'Г/А':'goal_contribution',
 'Г/А на натпревар':'gc_per_match',
 'Хет-трик ':'hat-trick',
 'Хет-трик од асистенции':'hat-trick-assists',
 'Автоголови':'owngoals',
 'Слободни':'freekicks',
 'Пенали':'penalty_stats',
 'Спасени пенали ':'penalty_saves',
 'Победи':'w',
 '% на победи':'w_percentage',
 'Порази':'losses',
 '% на порази':'loss_percentage',
 'Играч на натпреварот':'motm',
 'Номинации за играч на месец':'nominations',
 'Играч на месец':'potm',
 'Гол на месецот':'gotm'
}

In [265]:
columns = [columns_translating_mk_to_en.get(col) for col in columns]

In [266]:
columns

['index',
 'name',
 'position',
 'matches',
 'goals',
 'assists',
 'goal_contribution',
 'gc_per_match',
 'hat-trick',
 'hat-trick-assists',
 'owngoals',
 'freekicks',
 'penalty_stats',
 'penalty_saves',
 'w',
 'w_percentage',
 'losses',
 'loss_percentage',
 'motm',
 'nominations',
 'potm',
 'gotm']

In [267]:
df.columns = columns

In [268]:
df

,index,name,position,matches,goals,assists,goal_contribution,gc_per_match,hat-trick,hat-trick-assists,...,penalty_stats,penalty_saves,w,w_percentage,losses,loss_percentage,motm,nominations,potm,gotm
0,1,Дацо,D/GK,36,5,11,16,0.44,0,0,...,0/0,0/1,13,36%,16,44%,3,1,1,0
1,2,Лео,D/GK,34,40,19,59,1.74,3,3,...,0/1,1/2,12,35%,16,47%,3,2,3,1
2,3,Симиќ,AM,34,10,24,34,1.00,0,1,...,0/0,0/0,14,41%,16,47%,0,1,0,0
3,4,Стеф,M/GK,32,22,12,34,1.06,2,0,...,0/1,1/1,14,44%,12,38%,2,2,0,0
4,5,Нено,D/GK,29,0,6,6,0.21,0,0,...,0/1,1/3,11,38%,12,41%,0,2,0,0
5,6,Прем,A,27,38,19,57,2.11,7,0,...,1/1,0/0,11,41%,13,48%,3,2,0,4
6,7,Борјан,M/GK,27,31,19,50,1.85,4,1,...,0/1,0/0,17,63%,8,30%,0,0,1,0
7,8,Анчо,D,27,2,0,2,0.07,0,0,...,0/0,0/0,10,37%,13,48%,0,0,0,0
8,9,Хито,M,26,57,22,79,3.04,10,2,...,1/1,0/0,12,46%,10,38%,5,3,1,2
9,10,Христијан,D,25,14,21,35,1.40,0,1,...,0/1,0/0,13,52%,7,28%,1,1,1,0


In [269]:
split_positions = df['position'].str.split('/', expand=True)
df['position'] = split_positions[0]
df.insert(3, 'secondary_position', split_positions[1])

In [270]:
df

,index,name,position,secondary_position,matches,goals,assists,goal_contribution,gc_per_match,hat-trick,...,penalty_stats,penalty_saves,w,w_percentage,losses,loss_percentage,motm,nominations,potm,gotm
0,1,Дацо,D,GK,36,5,11,16,0.44,0,...,0/0,0/1,13,36%,16,44%,3,1,1,0
1,2,Лео,D,GK,34,40,19,59,1.74,3,...,0/1,1/2,12,35%,16,47%,3,2,3,1
2,3,Симиќ,AM,None,34,10,24,34,1.00,0,...,0/0,0/0,14,41%,16,47%,0,1,0,0
3,4,Стеф,M,GK,32,22,12,34,1.06,2,...,0/1,1/1,14,44%,12,38%,2,2,0,0
4,5,Нено,D,GK,29,0,6,6,0.21,0,...,0/1,1/3,11,38%,12,41%,0,2,0,0
5,6,Прем,A,None,27,38,19,57,2.11,7,...,1/1,0/0,11,41%,13,48%,3,2,0,4
6,7,Борјан,M,GK,27,31,19,50,1.85,4,...,0/1,0/0,17,63%,8,30%,0,0,1,0
7,8,Анчо,D,None,27,2,0,2,0.07,0,...,0/0,0/0,10,37%,13,48%,0,0,0,0
8,9,Хито,M,None,26,57,22,79,3.04,10,...,1/1,0/0,12,46%,10,38%,5,3,1,2
9,10,Христијан,D,None,25,14,21,35,1.40,0,...,0/1,0/0,13,52%,7,28%,1,1,1,0


In [271]:
df['loss_percentage'] = df['loss_percentage'].str.replace('%', '')
df['loss_percentage'] = df['loss_percentage'].astype(float) / 100
df['loss_percentage'] = df['loss_percentage'].round(decimals=2)

In [272]:
df['w_percentage'] = df['w_percentage'].str.replace('%', '')
df['w_percentage'] = df['w_percentage'].astype(float) / 100
df['w_percentage'] = df['w_percentage'].round(decimals=2)

In [273]:
df['name'] = df['name'].apply(cyrillicToLatinName)

In [274]:
df #inspect before save

,index,name,position,secondary_position,matches,goals,assists,goal_contribution,gc_per_match,hat-trick,...,penalty_stats,penalty_saves,w,w_percentage,losses,loss_percentage,motm,nominations,potm,gotm
0,1,Daco,D,GK,36,5,11,16,0.44,0,...,0/0,0/1,13,0.36,16,0.44,3,1,1,0
1,2,Leo,D,GK,34,40,19,59,1.74,3,...,0/1,1/2,12,0.35,16,0.47,3,2,3,1
2,3,Simikj,AM,None,34,10,24,34,1.00,0,...,0/0,0/0,14,0.41,16,0.47,0,1,0,0
3,4,Stef,M,GK,32,22,12,34,1.06,2,...,0/1,1/1,14,0.44,12,0.38,2,2,0,0
4,5,Neno,D,GK,29,0,6,6,0.21,0,...,0/1,1/3,11,0.38,12,0.41,0,2,0,0
5,6,Prem,A,None,27,38,19,57,2.11,7,...,1/1,0/0,11,0.41,13,0.48,3,2,0,4
6,7,Borjan,M,GK,27,31,19,50,1.85,4,...,0/1,0/0,17,0.63,8,0.30,0,0,1,0
7,8,Ancho,D,None,27,2,0,2,0.07,0,...,0/0,0/0,10,0.37,13,0.48,0,0,0,0
8,9,Hito,M,None,26,57,22,79,3.04,10,...,1/1,0/0,12,0.46,10,0.38,5,3,1,2
9,10,Hristijan,D,None,25,14,21,35,1.40,0,...,0/1,0/0,13,0.52,7,0.28,1,1,1,0


In [275]:
df.to_csv('zoki_liga_formatted.csv',index=False)

In [276]:
df_final = pd.read_csv('zoki_liga_final.csv')
df_test = pd.read_csv('zoki_liga.csv')
df_formatted = pd.read_csv('zoki_liga_formatted.csv')
#for test

In [277]:
# df_formatted.loc[df['secondary_position'].isna(), 'secondary_position'] = 'No Secondary Position'

In [278]:
# df_formatted

In [279]:
ratings_file_path = 'ratings/*.xlsx'
list_of_files = glob.glob(ratings_file_path)
print(list_of_files)

['ratings\\andrej_anakiev.xlsx', 'ratings\\borjan_todorovski.xlsx', 'ratings\\ivan_kuzmanovski.xlsx']


In [280]:
dataframes = []
for excel in list_of_files:
    df = pd.read_excel(excel)
    dataframes.append(df)

ID_COL = "name"
df_all = pd.concat(dataframes)
rating_cols = [c for c in df_all.columns if c not in {"index", "name"}]

#Average
df_avg = df_all.groupby(["index", "name"])[rating_cols].mean().round(2).sort_values("index")
#Mode
def first_mode(s: pd.Series): #mode
    m = s.mode(dropna=True)
    return m.iloc[0] if not m.empty else np.nan
df_mode = df_all.groupby(["index", "name"], as_index=False)[rating_cols].agg(first_mode).sort_values("index")

In [281]:
df_avg.to_csv("df_mean.csv", index=False)
df_mode.to_csv("df_mode.csv", index=False)

In [282]:
df_mode

,index,name,interceptions_overall_ability,fouls_commiting,stamina_rating,positions_played,leadership_measure,passing,physical,gk_ability,skill,positioning_sense
0,1,Daco,5.0,1.0,5.0,5.0,3.0,5.0,5.0,9.0,3.0,4.0
1,2,Leo,8.0,2.0,9.0,10.0,3.0,9.0,9.0,6.0,7.0,8.0
2,3,Simikj,4.0,1.0,3.0,4.0,2.0,8.0,3.0,1.0,8.0,6.0
3,4,Stef,3.0,3.0,4.0,3.0,4.0,3.0,2.0,2.0,2.0,4.0
4,5,Neno,2.0,7.0,5.0,2.0,1.0,2.0,7.0,7.0,2.0,2.0
5,6,Prem,6.0,2.0,6.0,6.0,6.0,7.0,6.0,1.0,8.0,8.0
6,7,Borjan,5.0,3.0,4.0,3.0,3.0,5.0,7.0,6.0,5.0,7.0
7,8,Ancho,3.0,1.0,6.0,2.0,2.0,2.0,3.0,1.0,2.0,5.0
8,9,Hito,8.0,10.0,7.0,5.0,5.0,8.0,6.0,1.0,8.0,9.0
9,10,Hristijan,4.0,4.0,3.0,4.0,1.0,3.0,7.0,1.0,4.0,4.0


In [283]:
df_avg

,,interceptions_overall_ability,fouls_commiting,stamina_rating,positions_played,leadership_measure,passing,physical,gk_ability,skill,positioning_sense
index,name,,,,,,,,,,
1,Daco,5.33,4.00,5.33,5.33,3.67,4.33,6.00,9.33,3.33,5.33
2,Leo,8.33,4.33,8.67,9.33,6.67,8.33,8.67,7.00,8.00,8.33
3,Simikj,5.67,3.33,4.00,4.00,3.00,7.67,2.33,1.00,7.67,7.00
4,Stef,4.67,2.33,5.00,5.33,5.33,4.33,5.00,5.00,4.00,5.00
5,Neno,3.00,8.00,5.33,3.00,1.00,3.67,6.33,7.67,3.00,2.67
6,Prem,7.00,3.00,6.00,5.33,6.33,7.33,7.00,2.67,8.33,8.00
7,Borjan,4.67,3.33,5.67,4.33,4.67,5.00,7.67,4.33,5.33,7.33
8,Ancho,3.67,5.33,5.00,2.00,3.00,2.33,3.33,1.33,2.33,4.33
9,Hito,8.33,8.67,8.67,5.33,6.00,7.67,7.67,2.67,8.33,8.33


In [284]:
df_combined = pd.concat([df_formatted.reset_index(drop=True),df_avg.reset_index(drop=True)], axis=1)

In [285]:
df_combined

,index,name,position,secondary_position,matches,goals,assists,goal_contribution,gc_per_match,hat-trick,...,interceptions_overall_ability,fouls_commiting,stamina_rating,positions_played,leadership_measure,passing,physical,gk_ability,skill,positioning_sense
0,1,Daco,D,GK,36,5,11,16,0.44,0,...,5.33,4.00,5.33,5.33,3.67,4.33,6.00,9.33,3.33,5.33
1,2,Leo,D,GK,34,40,19,59,1.74,3,...,8.33,4.33,8.67,9.33,6.67,8.33,8.67,7.00,8.00,8.33
2,3,Simikj,AM,NaN,34,10,24,34,1.00,0,...,5.67,3.33,4.00,4.00,3.00,7.67,2.33,1.00,7.67,7.00
3,4,Stef,M,GK,32,22,12,34,1.06,2,...,4.67,2.33,5.00,5.33,5.33,4.33,5.00,5.00,4.00,5.00
4,5,Neno,D,GK,29,0,6,6,0.21,0,...,3.00,8.00,5.33,3.00,1.00,3.67,6.33,7.67,3.00,2.67
5,6,Prem,A,NaN,27,38,19,57,2.11,7,...,7.00,3.00,6.00,5.33,6.33,7.33,7.00,2.67,8.33,8.00
6,7,Borjan,M,GK,27,31,19,50,1.85,4,...,4.67,3.33,5.67,4.33,4.67,5.00,7.67,4.33,5.33,7.33
7,8,Ancho,D,NaN,27,2,0,2,0.07,0,...,3.67,5.33,5.00,2.00,3.00,2.33,3.33,1.33,2.33,4.33
8,9,Hito,M,NaN,26,57,22,79,3.04,10,...,8.33,8.67,8.67,5.33,6.00,7.67,7.67,2.67,8.33,8.33
9,10,Hristijan,D,NaN,25,14,21,35,1.40,0,...,6.00,3.33,4.00,5.00,4.67,6.00,7.33,2.33,5.33,6.33


In [286]:
df_combined.to_csv("zoki_liga_formatted.csv", index=False)